# 01 - Data exploration

This notebook reproduces the exploratory analysis used in the project: loading `merged_dataset_with_generation.csv`, computing summary statistics, pairwise correlations, and generating plots saved to the `figures/` folder (correlation heatmap, collapse boxplots, and renewables penetration time series).

In [14]:
# Imports
%matplotlib inline
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure output folder
os.makedirs('figures', exist_ok=True)

In [15]:
# Load merged dataset (with generation)
p = '../merged_dataset_with_generation.csv'
df = pd.read_csv(p, parse_dates=['date'])
df = df.sort_values('date')
print('Loaded', p, 'shape=', df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../merged_dataset_with_generation.csv'

In [16]:
# Quick overview
display(df.head())
print('\nDatatypes:\n', df.dtypes)
print('\nMissing values per column:\n', df.isna().sum())

,date,price_spain,price_france,price_portugal,spread_fr_es,spread_pt_es,demand_mwh_day,demand_mean_mw,demand_peak_mw,demand_peak_hour,...,pressure_min_hpa,humidity_mean_pct,regime,is_collapse,year,month,day_of_year,quarter,is_weekend,gen_to_demand_ratio
0,2015-01-01,46.702500,41.499583,46.702500,-5.202917,0.00000,3440333.0,143347.208333,177444.0,21,...,996.5,69.0,normal,0,2015,1,1,1,0,0.051435
1,2015-01-02,54.322083,41.001250,54.322083,-13.320833,0.00000,4069521.0,169563.375000,203556.0,20,...,1000.5,60.0,normal,0,2015,1,2,1,0,0.059343
2,2015-01-03,53.768333,41.657500,53.768333,-12.110833,0.00000,3972525.0,165521.875000,197475.0,21,...,997.1,62.0,normal,0,2015,1,3,1,1,0.058577
3,2015-01-04,45.757083,34.417500,46.225833,-11.339583,0.46875,3777897.0,157412.375000,192147.0,21,...,995.9,78.0,normal,0,2015,1,4,1,1,0.054687
4,2015-01-05,59.776250,44.426250,59.776250,-15.350000,0.00000,4064704.0,169362.666667,197236.0,10,...,991.8,80.0,normal,0,2015,1,5,1,0,0.066249



Datatypes:
 date                      datetime64[ns]
price_spain                      float64
price_france                     float64
price_portugal                   float64
spread_fr_es                     float64
spread_pt_es                     float64
demand_mwh_day                   float64
demand_mean_mw                   float64
demand_peak_mw                   float64
demand_peak_hour                   int64
wind_gen_mwh_day                 float64
wind_gen_mean_mw                 float64
wind_gen_peak_mw                 float64
wind_gen_peak_hour                 int64
solar_pv_mwh_day                 float64
solar_pv_mean_mw                 float64
solar_pv_peak_mw                 float64
solar_pv_peak_hour                 int64
wind_penetration                 float64
solar_penetration                float64
renewables_penetration           float64
temp_mean_c                      float64
temp_max_c                       float64
temp_min_c                       float64
pre

## Pairwise correlations (selected variables)

In [17]:
vars_of_interest = [
    'price_spain','is_collapse','wind_penetration','solar_penetration','renewables_penetration',
    'gen_to_demand_ratio','temp_mean_c','wind_speed_ms','sunshine_hours'
]
corr = df[vars_of_interest].corr()
print(corr.round(3))
# Save CSV of correlations for reference
corr.to_csv('figures/selected_var_correlations.csv')
print('Saved correlations → figures/selected_var_correlations.csv')

                        price_spain  is_collapse  wind_penetration  \
price_spain                   1.000       -0.241            -0.282   
is_collapse                  -0.241        1.000             0.161   
wind_penetration             -0.282        0.161             1.000   
solar_penetration            -0.221       -0.161             0.070   
renewables_penetration       -0.329       -0.044             0.590   
gen_to_demand_ratio          -0.329       -0.044             0.590   
temp_mean_c                  -0.082       -0.069            -0.305   
wind_speed_ms                -0.196        0.147            -0.038   
sunshine_hours               -0.119       -0.025            -0.100   

                        solar_penetration  renewables_penetration  \
price_spain                        -0.221                  -0.329   
is_collapse                        -0.161                  -0.044   
wind_penetration                    0.070                   0.590   
solar_penetration      

In [18]:
# Heatmap (matplotlib, no seaborn dependency)
plt.figure(figsize=(8,6))
mat = plt.matshow(corr, fignum=1, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(mat)
plt.xticks(range(len(vars_of_interest)), vars_of_interest, rotation=45, ha='left')
plt.yticks(range(len(vars_of_interest)), vars_of_interest)
# annotate values
import numpy as _np
for (i,j), val in _np.ndenumerate(corr.values):
    plt.text(j, i, f"{val:.3f}", ha='center', va='center', fontsize=8)
plt.title('Correlation matrix (selected vars)')
plt.savefig('figures/corr_heatmap.png', dpi=150)
plt.close()
print('Saved heatmap → figures/corr_heatmap.png')

Saved heatmap → figures/corr_heatmap.png


## Collapse vs Normal: boxplots for top predictors

In [12]:
# Compute absolute correlations with collapse and plot top 4
corr_with_collapse = corr['is_collapse'].drop('is_collapse').abs().sort_values(ascending=False)
print('Absolute correlations with is_collapse:')
print(corr_with_collapse.round(3))
top_vars = list(corr_with_collapse.index[:4])
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(top_vars),1, figsize=(10,4*len(top_vars)))
for i,v in enumerate(top_vars):
    ax = axes[i] if len(top_vars)>1 else axes
    data0 = df.loc[df['is_collapse']==0, v].dropna()
    data1 = df.loc[df['is_collapse']==1, v].dropna()
    ax.boxplot([data0, data1], tick_labels=['normal','collapse'])
    ax.set_title(f'{v} by collapse')
    ax.set_ylabel(v)
plt.tight_layout()
plt.savefig('figures/collapse_boxplots.png', dpi=150)
plt.close()
print('Saved boxplots → figures/collapse_boxplots.png')

Absolute correlations with is_collapse:
price_spain               0.241
wind_penetration          0.161
solar_penetration         0.161
wind_speed_ms             0.147
temp_mean_c               0.069
gen_to_demand_ratio       0.044
renewables_penetration    0.044
sunshine_hours            0.025
Name: is_collapse, dtype: float64
Saved boxplots → figures/collapse_boxplots.png


## Time series: renewables penetration (30-day moving average)

In [13]:
df_ts = df.set_index('date').sort_index()
plt.figure(figsize=(12,4))
df_ts['renewables_penetration'].rolling(30, min_periods=1).mean().plot()
plt.title('30-day moving average: renewables_penetration')
plt.ylabel('renewables_penetration')
plt.tight_layout()
plt.savefig('figures/renewables_penetration_ts.png', dpi=150)
plt.close()
print('Saved time series → figures/renewables_penetration_ts.png')

Saved time series → figures/renewables_penetration_ts.png


### Notes
- Plots are saved to the `figures/` directory. If you want inline plots in the notebook, add `%matplotlib inline` and display the figures instead of saving.
- This notebook uses only the merged dataset; upstream imputation is in `02_data_imputation.ipynb`.